In [ ]:
import os
import re
import glob
import time
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import pandas as pd

from matplotlib.ticker import AutoMinorLocator
from scipy.signal import argrelextrema

from JunoPy.mag import MagData, AutoFluxRope
from JunoPy.library import find_ticks_between


In [ ]:
# start = dt.datetime.strptime('2017-236T19:11:00', '%Y-%jT%H:%M:%S') - dt.timedelta(minutes=10)
# end = dt.datetime.strptime('2017-236T21:21:00', '%Y-%jT%H:%M:%S') + dt.timedelta(minutes=10)

# start = dt.datetime.strptime('2017-175T00:11:00', '%Y-%jT%H:%M:%S') - dt.timedelta(minutes=10)
# end = dt.datetime.strptime('2017-175T23:48:00', '%Y-%jT%H:%M:%S') + dt.timedelta(minutes=10)

start = dt.datetime.strptime('2017-338T00:48:57', '%Y-%jT%H:%M:%S')
end = dt.datetime.strptime('2017-338T02:51:59', '%Y-%jT%H:%M:%S')
filename = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]

magdata = MagData(filename=filename)
# magdata.df.head()

In [ ]:
crossings = AutoFluxRope.find_btheta_crossings(magdata) #, start, end)
ts, te = AutoFluxRope.find_plasmoid_candidates(magdata, crossings, window=5)

# Plot the data.

In [ ]:
def plot_magdata(magdata, start, end, crossings, ts, te):
    """Plot data between start and end."""
    
    # Slice the magnetometer data.
    # magdata.df = magdata.df.loc[start:end]
    
    # Find the ticks.
    timearray, tickstring = find_ticks_between(start, end)
    
    # Create the figure.
    fig, axs = plt.subplots(4, 1, 
                            gridspec_kw={'hspace': 0, 'right':0.9, 'top':0.9}, 
                            figsize=(11, 8.5), 
                            dpi=300)
    
    axs[:] = magdata.plot(vars=['Br', 'Btheta', 'Bphi', 'Bmag'], axs=axs)
    
    for point in crossings:
        df_crossings = magdata.df.loc[crossings].copy()
        axs[1].scatter(df_crossings.index, df_crossings['Btheta'].values, color='g')
        
    for point in ts:
        df_start = magdata.df.loc[ts].copy()
        axs[1].scatter(df_start.index, df_start['Btheta'].values, color='r', alpha=0.5)
        
    for point in te:
        df_end = magdata.df.loc[te].copy()
        axs[1].scatter(df_end.index, df_end['Btheta'].values, color='b', alpha=0.5)
    
    for a in axs:
        a.set_xticks(timearray)
        a.set_xticklabels(tickstring)
        a.set_xlim([timearray[0], timearray[-1]])

        a.label_outer()
    a.tick_params(axis='x', width=2, rotation=0)
    a.xaxis.label.set_visible(False)
    
    plt.suptitle(dt.datetime.strftime(start, '%Y-%m-%d') + \
                ', DOY={:s}'.format(dt.datetime.strftime(start, '%j')), y=0.92)
    
    # fig.savefig('out_auto_fluxrope/' + dt.datetime.strftime(start, '%Y-%jT%H:%M') + '.png')
    # plt.close(fig)
    
# plot_magdata(magdata, start, end, cs, ts, te)


In [ ]:
def plot_magdata_line(magdata, start, end, ts, te):
    """Plot data between start and end."""
    
    # Slice the magnetometer data.
    # magdata.df = magdata.df.loc[start:end]
    
    # Find the ticks.
    timearray, tickstring = find_ticks_between(start, end)
    
    # Create the figure.
    fig, axs = plt.subplots(4, 1, 
                            gridspec_kw={'hspace': 0, 'right':0.9, 'top':0.9}, 
                            figsize=(11, 8.5), 
                            dpi=300)
    
    axs[:] = magdata.plot(vars=['Br', 'Btheta', 'Bphi', 'Bmag'], axs=axs)
        
    for tstart, tend in zip(ts, te):
        df_line = magdata.df.loc[tstart:tend].copy()
        if df_line.loc[tstart, 'Btheta'] < 0:
            axs[1].plot(df_line.index, df_line['Btheta'].values, color='b')
        else:
            axs[1].plot(df_line.index, df_line['Btheta'].values, color='r')
    
    for a in axs:
        a.set_xticks(timearray)
        a.set_xticklabels(tickstring)
        a.set_xlim([timearray[0], timearray[-1]])

        a.label_outer()
    a.tick_params(axis='x', width=2, rotation=0)
    a.xaxis.label.set_visible(False)
    
    plt.suptitle(dt.datetime.strftime(start, '%Y-%m-%d') + \
                ', DOY={:s}'.format(dt.datetime.strftime(start, '%j')), y=0.92)
    
    fig.savefig('out_auto_fluxrope/' + dt.datetime.strftime(start, '%Y-%jT%H:%M') + '_lines.png')
    # plt.close(fig)
    
plot_magdata_line(magdata, start, end, ts, te)


In [ ]:
ts, te